In [1]:
import datasets
import numpy as np
import os
from datasets import ClassLabel
import pandas as pd 
import torch
import os
from torch import optim, nn, utils
from torch.utils.data import Dataset
import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint
#from lightning.pytorch.loggers.tensorboard import TensorBoardLogger
from lightning.pytorch.callbacks.early_stopping import EarlyStopping
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report
from torchmetrics.classification import ConfusionMatrix
import argparse
import random
import math
import matplotlib.pyplot as plt

/work/art-multimodal-benchmark/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# progress is still (for subclassification task, specifically):

# filtering dataframe for specific task + remapping labels for hf classLabel feature

# NO SPLITTING ! 

# k-fold crossvalidation splitting:

    # create dataloaders for k split

    # build model

    # train

    # validate

    # repeat


In [8]:
def remap_features(ds_original, ds_filtered, label):

    original_feature = ds_original.features[label] # the ClassLabel feature
    original_names = original_feature.names

    # classes(names) in new subclassification dataset:
    used_class_names = sorted(list(set(ds_filtered[f"{label}_str"])))
    new_class_label = ClassLabel(names=used_class_names)

    # set up function to remap from str -> int for new ClassLabels
    def remap_labels(example):
        example[label] = new_class_label.str2int(example[f"{label}_str"])
        return example
    
    # use map to remap classlabels
    ds_filtered = ds_filtered.map(remap_labels)

    # recast the class label feature to new labels
    new_features = ds_filtered.features.copy()
    new_features[label] = new_class_label
    ds_filtered = ds_filtered.cast(new_features)

    return ds_filtered

def filter_data(ds, label, subclassification_task):
    ds = ds.add_column('old_indices', range(len(ds)))

    # find the rows that matches the subclassification task
    subclass_indices = [idx for idx, a in enumerate(ds[f'{label}_str']) if a in subclassification_task]
    ds_subset = ds.select(subclass_indices)

    # remap labels to fit to new number of classes for subclassification task
    ds_subset = remap_features(ds, ds_subset, label)

    ds_split = ds_subset.train_test_split(test_size=0.2, seed=2830, stratify_by_column=label)

    return ds_split['train'], ds_split['test']

In [10]:
ds = datasets.load_from_disk(os.path.join('..', 'data', 'wikiart_filtered_remapped_FINAL'))

In [11]:
subclassification_task = ['Expressionism', 'Impressionism', 'Post_Impressionism']

ds_train_val, ds_test = filter_data(ds, 'style', subclassification_task)

del ds

In [ ]:
from sklearn.model_selection import StratifiedKFold

label= 'style'
skf = StratifiedKFold(n_splits = 5, shuffle=True, random_state=2830) # shuffle=true?

# fetch labels

labels = np.array(ds_train_val[label])
indices = np.arange(len(ds_train_val))

# still need to understand this syntax completely
for fold, (train_idx, val_idx) in enumerate(skf.split(indices, labels)):
    
    # monitor splits
    print(f"Fold {fold+1}")

    ds_train = ds_train_val.select(train_idx.tolist())
    ds_val = train_val_ds.select(val_idx.tolist())

    ds_splits = {
        'train': ds_train,
        'val': ds_val
    }

    




[    0     1     2 ... 11212 11213 11214]
[    5     6     7 ... 11205 11208 11215]
0
[    0     1     2 ... 11213 11214 11215]
[    9    10    12 ... 11200 11207 11210]
1
[    0     1     2 ... 11211 11212 11215]
[    3     4    15 ... 11199 11213 11214]
2
[    0     3     4 ... 11213 11214 11215]
[    1     2    11 ... 11202 11206 11211]
3
[    1     2     3 ... 11213 11214 11215]
[    0    13    18 ... 11204 11209 11212]
4
